### Creating Dim_Date dimention table based on silver.order_items, silver.orders & silver.reviews table

In [0]:
from pyspark.sql import functions as F

In [0]:
df_ot = spark.table("silver.order_items")
df_ot.printSchema()

In [0]:
df_o = spark.table("silver.orders")
df_o.printSchema()

In [0]:
df_r = spark.table("silver.reviews")
df_r.printSchema()

## 1. Collect all date/timestamp columns that feed facts in Silver

In [0]:
date_cols = [
    ("silver.order_items", "shipping_limit_date"),
    ("silver.orders", "order_purchase_timestamp"),
    ("silver.orders", "order_approved_at"),
    ("silver.orders", "order_delivered_carrier_date"),
    ("silver.orders", "order_delivered_customer_date"),
    ("silver.orders", "order_estimated_delivery_date"),
    ("silver.reviews", "review_creation_date"),
    ("silver.reviews", "review_answer_timestamp"),
]

## 2. Reduce each table to its local min/max, then combine

In [0]:
(
    df_o.select(
            F.min(F.to_date("order_approved_at")).alias("min_d"), 
            F.max(F.to_date("order_approved_at")).alias("max_d")
    ).first()
)

In [0]:
import builtins
bounds = []
for table, col in date_cols:
    row = (
        spark.table(table)
        .select(
            F.min(F.to_date(F.col(col))).alias("min_d"),
            F.max(F.to_date(F.col(col))).alias("max_d"),
        )
        .first()
    )
    if row.min_d is not None:
        bounds.append((row.min_d, row.max_d))

global_min = builtins.min(b[0] for b in bounds)
global_max = builtins.max(b[1] for b in bounds)



## 3. Pad to clean year boundaries so partial years are fully covered

In [0]:

start_date = global_min.replace(month=1, day=1)
end_date   = global_max.replace(month=12, day=31)

print(f"Date spine range: {start_date} -> {end_date}")

# NOW dim_date table generation

### 1. Generate the date spine

In [0]:
# date_spine = spark.sql(f"""
#     SELECT explode(
#         sequence(to_date('{start_date}'), to_date('{end_date}'), interval 1 day)
#     ) AS date
# """)

# OR
date_spine = spark.range(1).select(
    F.explode(
        F.sequence(F.lit(start_date), F.lit(end_date), F.expr("interval 1 day"))
    ).alias("date")
)

In [0]:
date_spine.first()

In [0]:
date_spine.limit(5).display()

### 2. Derive attributes

In [0]:
date_dim = (
    date_spine
    .withColumn("date_key",        F.date_format("date", "yyyyMMdd").cast("int"))
    .withColumn("year",            F.year("date"))
    .withColumn("quarter",         F.quarter("date"))
    .withColumn("month",           F.month("date"))
    .withColumn("month_name",      F.date_format("date", "MMMM"))
    .withColumn("day",             F.dayofmonth("date"))
    .withColumn("day_of_week",     F.dayofweek("date"))           # 1=Sun..7=Sat
    .withColumn("day_name",        F.date_format("date", "EEEE"))
    .withColumn("day_of_year",     F.dayofyear("date"))
    .withColumn("week_of_year",    F.weekofyear("date"))
    .withColumn("is_weekend",      F.dayofweek("date").isin(1, 7))
    .withColumn("year_month",      F.date_format("date", "yyyy-MM"))
    .withColumn("quarter_name",    F.concat(F.lit("Q"), F.quarter("date")))
    .withColumn("first_day_of_month", F.trunc("date", "month"))
    .withColumn("last_day_of_month",  F.last_day("date"))
)


In [0]:
date_dim.limit(5).display()

### 3. Write to Gold as a managed Delta table


In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS gold")

In [0]:
(
    date_dim.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("gold.dim_date")
)

In [0]:
spark.table("gold.dim_date").limit(5).display()